In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
df = pd.read_csv("data/raw.csv", low_memory=False)
df.columns = df.columns.str.strip()

print(df.shape)
df.head()

(692703, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80,38308,1,1,6,6,6,6,6.000000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,389,479,11,5,172,326,79,0,15.636364,31.449238,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,88,1095,10,6,3150,3150,1575,0,315.000000,632.561635,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,389,15206,17,12,3452,6660,1313,0,203.058823,425.778474,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,88,1092,9,6,3150,3152,1575,0,350.000000,694.509719,...,32,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [3]:
df["Attack"] = (df["Label"] != "BENIGN").astype(int)

df["Attack"].value_counts()

Attack
0    440031
1    252672
Name: count, dtype: int64

In [4]:
X = df.drop(columns=["Label", "Attack"])
X = X.select_dtypes(include="number")
X = X.replace([np.inf, -np.inf], np.nan)

y = df["Attack"]

print(X.shape)

(692703, 78)


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(554162, 78)
(138541, 78)


In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)

## Model Output Interpretation

The anomaly detection model assigns scores to each network flow.
Lower scores indicate flows that deviate more strongly from normal traffic behavior.

## Security Interpretation

The lowest anomaly scores represent network flows that deviate most
from the baseline learned from benign traffic.

In an operational security environment these flows would not
automatically be classified as attacks. Instead they would be
prioritized for analyst review and correlated with other signals
such as firewall logs, endpoint activity, or SIEM alerts.

Anomaly detection helps surface unusual traffic patterns that may
indicate scanning activity, denial-of-service traffic, or other
suspicious behavior.

In [ ]:
# Visualize the most suspicious network flows

import pandas as pd
import matplotlib.pyplot as plt

df_scores = pd.read_csv("outputs/top_anomalies.csv").sort_values("anomaly_score")

df_scores = df_scores.head(20)

plt.figure(figsize=(10,5))
plt.bar(range(len(df_scores)), df_scores["anomaly_score"])

plt.title("Top Suspicious Network Flows")
plt.xlabel("Flow Rank")
plt.ylabel("Anomaly Score")

plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()